In [1]:
from langgraph.graph import StateGraph,START ,END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage , HumanMessage , AIMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv 
load_dotenv()
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver


In [2]:
class ChatbotState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [3]:
model=ChatOpenAI()

def chatbot(state:ChatbotState):
    message=state["messages"]
    response=model.invoke(message)
    return {"messages":[response]}



In [4]:
checkpoint=MemorySaver()
graph=StateGraph(ChatbotState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START,"chatbot")
graph.add_edge("chatbot",END)
app=graph.compile(checkpointer=checkpoint)
initial_state={"messages":[HumanMessage(content="Hello,how are you doing?")]}




In [9]:
thread_id = "1"
config = {"configurable": {"thread_id": thread_id}}

while True:
    user_input = input("User: ")
    print("User:", user_input)
    
    if user_input.strip().lower() in ["exit", "quit", "bye"]:
        print("Chatbot: Goodbye!")
        break

   
    print("Chatbot: ", end="", flush=True)
    
 
    input_data = {"messages": [HumanMessage(content=user_input)]}
    
    for message, metadata in app.stream(input_data, config=config, stream_mode="messages"):
        if message.content:
            
            print(message.content, end="", flush=True)
    
    print() 

User: hi
Chatbot: Hello! How can I assist you today?
User: how are you 
Chatbot: I'm just a computer program, so I don't have feelings, but I'm here to help you with any questions you have. How can I assist you today?
User: exit
Chatbot: Goodbye!


In [30]:
app.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='3c77be52-b3d6-43da-9383-245575e61b52'), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DWw2HS2KXpZUJVDAr0fZrUk2pKpgn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dae12-6347-7d83-a041-c63196f9df60-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details